# 51 — Modern Domain-Adversarial and Invariant Learning Baselines

**Purpose:** Test whether learned invariant representations can beat the failure of
classical feature engineering and transforms. This addresses a key reviewer expectation.

**Output directory:** `artifacts/thesis_finalization/nb51_modern_domain_adaptation/`

### Methods tested:
1. DANN-style domain-adversarial baseline (gradient reversal)
2. MMD-regularized representation baseline
3. CORAL-loss training baseline
4. IRM-inspired invariant risk baseline

### What changes relative to earlier notebooks?
This is a **new** notebook. NB48 includes a pooled evaluation of these methods;
NB51 adds LODO evaluation, training curves, and latent space analysis.

## 0. Setup

In [1]:
import sys, json, warnings, os
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.15)
np.random.seed(42)

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.clean_pipeline.feature_families import SAFE_CORE_PLUS_TEMPORAL
CLEAN = ROOT / "artifacts" / "clean_pipeline"
OUT   = ROOT / "artifacts" / "thesis_finalization" / "nb51_modern_domain_adaptation"
OUT.mkdir(parents=True, exist_ok=True)
FEAT_COLS = list(SAFE_CORE_PLUS_TEMPORAL)
SEED = 42; EPS = 1e-9; TIMESTAMP = datetime.now().isoformat()

def save_json(obj, n):
    p = OUT / n; open(p,"w").write(json.dumps(obj,indent=2,default=str)); print(f"  ✓ {p}")
def save_md(t, n):
    (OUT / n).write_text(t, encoding="utf-8"); print(f"  ✓ {OUT/n}")
def save_csv(d, n):
    p = OUT / n; (d if isinstance(d,pd.DataFrame) else pd.DataFrame(d)).to_csv(p); print(f"  ✓ {p}")
def save_fig(f, n, dpi=200):
    p = OUT / n; f.savefig(p, dpi=dpi, bbox_inches="tight", facecolor="white"); plt.close(f); print(f"  ✓ {p}")

df = pd.read_parquet(CLEAN / "features.parquet")
DATASETS = sorted(df["dataset"].unique())
le = LabelEncoder(); df["ds_enc"] = le.fit_transform(df["dataset"])
n_domains = len(DATASETS)
print(f"Loaded {len(df):,} flows, {len(DATASETS)} datasets")

Loaded 72,612 flows, 3 datasets


## 1. PyTorch Setup and Data Preparation

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Standard-scale features
scaler = StandardScaler()
train_idx = df["split"] == "train"
test_idx = df["split"] == "test"
X_all_scaled = scaler.fit_transform(df[FEAT_COLS].values)
df_scaled = df.copy()
for i, f in enumerate(FEAT_COLS):
    df_scaled[f] = X_all_scaled[:, i]

n_features = len(FEAT_COLS)
print(f"Features: {n_features}, Device: {device}")

Device: cpu
Features: 21, Device: cpu


## 2. Model Architectures

In [3]:
class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

class SharedEncoder(nn.Module):
    def __init__(self, n_in, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Dropout(0.3),
                                  nn.Linear(hidden, hidden//2), nn.ReLU())
    def forward(self, x):
        return self.net(x)

class VPNHead(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, 1), nn.Sigmoid())
    def forward(self, x):
        return self.net(x).squeeze(-1)

class DomainHead(nn.Module):
    def __init__(self, n_in, n_domains):
        super().__init__()
        self.net = nn.Linear(n_in, n_domains)
    def forward(self, x):
        return self.net(x)

def evaluate_model(encoder, vpn_head, X, y, d):
    """Evaluate VPN detection and latent domain separability."""
    encoder.eval(); vpn_head.eval()
    with torch.no_grad():
        feats = encoder(X.to(device))
        preds = vpn_head(feats).cpu().numpy()
        feats_np = feats.cpu().numpy()
    y_np = y.numpy(); d_np = d.numpy()
    
    results = {}
    if len(np.unique(y_np)) == 2:
        results["vpn_auc"] = float(roc_auc_score(y_np, preds))
    else:
        results["vpn_auc"] = np.nan
    
    # Latent domain AUC
    if len(np.unique(d_np)) >= 2:
        dc = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=SEED)
        dc.fit(feats_np, d_np)
        try:
            results["latent_domain_auc"] = float(roc_auc_score(
                d_np, dc.predict_proba(feats_np), multi_class="ovr", average="macro"))
        except:
            results["latent_domain_auc"] = np.nan
    else:
        results["latent_domain_auc"] = np.nan
    return results, preds, feats_np

def train_loop(encoder, vpn_head, X_tr, y_tr, d_tr, loss_extra_fn, epochs=150, lr=1e-3):
    """Generic training loop with pluggable extra loss."""
    optimizer = optim.Adam(list(encoder.parameters()) + list(vpn_head.parameters()), lr=lr)
    bce = nn.BCELoss()
    dataset = TensorDataset(X_tr.to(device), y_tr.to(device), d_tr.to(device))
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    history = []
    
    for epoch in range(epochs):
        encoder.train(); vpn_head.train()
        epoch_loss = 0; n_batches = 0
        for xb, yb, db in loader:
            feats = encoder(xb)
            vpn_pred = vpn_head(feats)
            loss_vpn = bce(vpn_pred, yb)
            loss_extra = loss_extra_fn(feats, xb, yb, db, epoch, epochs)
            loss = loss_vpn + loss_extra
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            epoch_loss += loss.item(); n_batches += 1
        history.append({"epoch": epoch, "loss": epoch_loss / max(n_batches,1)})
    return history
print("Model architectures defined.")

Model architectures defined.


---
## 3. DANN — Domain-Adversarial Network

In [4]:
def run_dann(X_tr, y_tr, d_tr, X_te, y_te, d_te, lambda_d=0.5, epochs=150):
    # Re-encode domain labels to 0..n-1 for this subset
    all_d = torch.cat([d_tr, d_te])
    uniq_d = torch.unique(all_d)
    d_map = {int(v): i for i, v in enumerate(uniq_d)}
    d_tr_enc = torch.LongTensor([d_map[int(v)] for v in d_tr])
    d_te_enc = torch.LongTensor([d_map[int(v)] for v in d_te])
    n_dom_local = len(uniq_d)

    hidden = 32; h2 = hidden // 2
    enc = SharedEncoder(n_features, hidden).to(device)
    vpn_h = VPNHead(h2).to(device)
    dom_h = DomainHead(h2, n_dom_local).to(device)

    optimizer = optim.Adam(list(enc.parameters()) + list(vpn_h.parameters()) + list(dom_h.parameters()), lr=1e-3)
    bce = nn.BCELoss(); ce = nn.CrossEntropyLoss()
    dataset = TensorDataset(X_tr.to(device), y_tr.to(device), d_tr_enc.to(device))
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    history = []
    
    for epoch in range(epochs):
        enc.train(); vpn_h.train(); dom_h.train()
        p = float(epoch) / epochs
        alpha = 2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0
        ep_loss = 0; nb = 0
        for xb, yb, db in loader:
            feats = enc(xb)
            loss_vpn = bce(vpn_h(feats), yb)
            rev_feats = GradientReversalLayer.apply(feats, alpha)
            loss_dom = ce(dom_h(rev_feats), db)
            loss = loss_vpn + lambda_d * loss_dom
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            ep_loss += loss.item(); nb += 1
        history.append({"epoch": epoch, "loss": ep_loss/max(nb,1)})
    
    results, preds, feats = evaluate_model(enc, vpn_h, X_te, y_te, d_te_enc)
    return results, history, preds, feats

# Run DANN on pooled data
X_tr_t = torch.FloatTensor(df_scaled.loc[train_idx, FEAT_COLS].values)
y_tr_t = torch.FloatTensor(df_scaled.loc[train_idx, "label"].values)
d_tr_t = torch.LongTensor(df_scaled.loc[train_idx, "ds_enc"].values)
X_te_t = torch.FloatTensor(df_scaled.loc[test_idx, FEAT_COLS].values)
y_te_t = torch.FloatTensor(df_scaled.loc[test_idx, "label"].values)
d_te_t = torch.LongTensor(df_scaled.loc[test_idx, "ds_enc"].values)

print("Training DANN (pooled)...")
dann_res, dann_hist, dann_p, dann_f = run_dann(X_tr_t, y_tr_t, d_tr_t, X_te_t, y_te_t, d_te_t)
print(f"  VPN AUC: {dann_res['vpn_auc']:.4f}, Latent Domain AUC: {dann_res['latent_domain_auc']:.4f}")

Training DANN (pooled)...
  VPN AUC: 0.9392, Latent Domain AUC: 0.9988


In [5]:
# LODO evaluation for DANN
dann_lodo = []
for test_ds in DATASETS:
    train_ds = [d for d in DATASETS if d != test_ds]
    tr_mask = (df_scaled["dataset"].isin(train_ds)) & (df_scaled["split"] == "train")
    te_mask = df_scaled["dataset"] == test_ds
    
    if tr_mask.sum() == 0 or te_mask.sum() == 0: continue
    if len(np.unique(df_scaled.loc[tr_mask, "label"])) < 2: continue
    
    Xtr = torch.FloatTensor(df_scaled.loc[tr_mask, FEAT_COLS].values)
    ytr = torch.FloatTensor(df_scaled.loc[tr_mask, "label"].values)
    dtr = torch.LongTensor(df_scaled.loc[tr_mask, "ds_enc"].values)
    Xte = torch.FloatTensor(df_scaled.loc[te_mask, FEAT_COLS].values)
    yte = torch.FloatTensor(df_scaled.loc[te_mask, "label"].values)
    dte = torch.LongTensor(df_scaled.loc[te_mask, "ds_enc"].values)
    
    res, _, _, _ = run_dann(Xtr, ytr, dtr, Xte, yte, dte, epochs=100)
    dann_lodo.append({"held_out": test_ds, **res})
    print(f"  DANN LODO test={test_ds}: vpn_auc={res['vpn_auc']:.4f}")

dann_lodo_df = pd.DataFrame(dann_lodo)
print(f"DANN LODO min AUC: {dann_lodo_df['vpn_auc'].min():.4f}")

  DANN LODO test=iscx: vpn_auc=0.4058
  DANN LODO test=usbvpn: vpn_auc=0.2158
  DANN LODO test=vnat: vpn_auc=0.3647
DANN LODO min AUC: 0.2158


---
## 4. MMD-Regularized Baseline

In [7]:
def compute_mmd(x, y, sigma=1.0):
    xx = torch.mm(x, x.t()); yy = torch.mm(y, y.t()); xy = torch.mm(x, y.t())
    rx = xx.diag().unsqueeze(0).expand_as(xx); ry = yy.diag().unsqueeze(0).expand_as(yy)
    Kxx = torch.exp(-sigma * (rx.t() + rx - 2*xx))
    Kyy = torch.exp(-sigma * (ry.t() + ry - 2*yy))
    # Use proper broadcasting for cross-term: [nx,1] + [1,ny] -> [nx,ny]
    Kxy = torch.exp(-sigma * (xx.diag().unsqueeze(1) + yy.diag().unsqueeze(0) - 2*xy))
    return Kxx.mean() + Kyy.mean() - 2*Kxy.mean()

def run_mmd(X_tr, y_tr, d_tr, X_te, y_te, d_te, lambda_mmd=0.1, epochs=150):
    # Re-encode domain labels
    all_d = torch.cat([d_tr, d_te])
    uniq_d = torch.unique(all_d)
    d_map = {int(v): i for i, v in enumerate(uniq_d)}
    d_tr_enc = torch.LongTensor([d_map[int(v)] for v in d_tr])
    d_te_enc = torch.LongTensor([d_map[int(v)] for v in d_te])

    enc = SharedEncoder(n_features, 32).to(device)
    vpn_h = VPNHead(16).to(device)
    optimizer = optim.Adam(list(enc.parameters())+list(vpn_h.parameters()), lr=1e-3)
    bce = nn.BCELoss()
    dataset = TensorDataset(X_tr.to(device), y_tr.to(device), d_tr_enc.to(device))
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    history = []; d_np = d_tr_enc.numpy(); domain_ids = np.unique(d_np)

    for epoch in range(epochs):
        enc.train(); vpn_h.train(); el = 0; nb = 0
        for xb, yb, db in loader:
            feats = enc(xb); loss_cls = bce(vpn_h(feats), yb)
            db_np = db.cpu().numpy()
            loss_mmd = torch.tensor(0.0, device=device); np_ = 0
            for i in range(len(domain_ids)):
                for j in range(i+1, len(domain_ids)):
                    mi = torch.BoolTensor(db_np==domain_ids[i]).to(device)
                    mj = torch.BoolTensor(db_np==domain_ids[j]).to(device)
                    if mi.sum()>1 and mj.sum()>1:
                        loss_mmd = loss_mmd + compute_mmd(feats[mi][:80], feats[mj][:80]); np_ += 1
            if np_>0: loss_mmd /= np_
            loss = loss_cls + lambda_mmd * loss_mmd
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            el += loss.item(); nb += 1
        history.append({"epoch":epoch,"loss":el/max(nb,1)})
    return evaluate_model(enc, vpn_h, X_te, y_te, d_te_enc)[0], history

print("Training MMD (pooled)...")
mmd_res, mmd_hist = run_mmd(X_tr_t, y_tr_t, d_tr_t, X_te_t, y_te_t, d_te_t)
print(f"  VPN AUC: {mmd_res['vpn_auc']:.4f}, Latent Domain AUC: {mmd_res['latent_domain_auc']:.4f}")

mmd_lodo = []
for test_ds in DATASETS:
    train_ds = [d for d in DATASETS if d != test_ds]
    tr_mask = (df_scaled["dataset"].isin(train_ds)) & (df_scaled["split"]=="train")
    te_mask = df_scaled["dataset"]==test_ds
    if tr_mask.sum()==0 or te_mask.sum()==0: continue
    if len(np.unique(df_scaled.loc[tr_mask,"label"]))<2: continue
    Xtr=torch.FloatTensor(df_scaled.loc[tr_mask,FEAT_COLS].values)
    ytr=torch.FloatTensor(df_scaled.loc[tr_mask,"label"].values)
    dtr=torch.LongTensor(df_scaled.loc[tr_mask,"ds_enc"].values)
    Xte=torch.FloatTensor(df_scaled.loc[te_mask,FEAT_COLS].values)
    yte=torch.FloatTensor(df_scaled.loc[te_mask,"label"].values)
    dte=torch.LongTensor(df_scaled.loc[te_mask,"ds_enc"].values)
    res, _ = run_mmd(Xtr,ytr,dtr,Xte,yte,dte,epochs=100)
    mmd_lodo.append({"held_out":test_ds,**res})
    print(f"  MMD LODO test={test_ds}: vpn_auc={res['vpn_auc']:.4f}")
mmd_lodo_df = pd.DataFrame(mmd_lodo)

Training MMD (pooled)...
  VPN AUC: 0.9245, Latent Domain AUC: 0.9993
  MMD LODO test=iscx: vpn_auc=0.4171
  MMD LODO test=usbvpn: vpn_auc=0.2387
  MMD LODO test=vnat: vpn_auc=0.2075


---
## 5. CORAL-Loss Training Baseline

In [8]:
def coral_loss(s, t):
    d = s.shape[1]; ns = s.shape[0]; nt = t.shape[0]
    cs = ((s - s.mean(0)).t() @ (s - s.mean(0))) / max(ns-1,1)
    ct = ((t - t.mean(0)).t() @ (t - t.mean(0))) / max(nt-1,1)
    return ((cs - ct)**2).sum() / (4*d*d)

def run_coral(X_tr, y_tr, d_tr, X_te, y_te, d_te, lam=0.1, epochs=150):
    # Re-encode domain labels
    all_d = torch.cat([d_tr, d_te])
    uniq_d = torch.unique(all_d)
    d_map = {int(v): i for i, v in enumerate(uniq_d)}
    d_tr_enc = torch.LongTensor([d_map[int(v)] for v in d_tr])
    d_te_enc = torch.LongTensor([d_map[int(v)] for v in d_te])

    enc = SharedEncoder(n_features, 32).to(device)
    vpn_h = VPNHead(16).to(device)
    opt = optim.Adam(list(enc.parameters())+list(vpn_h.parameters()), lr=1e-3)
    bce = nn.BCELoss()
    ds_ = TensorDataset(X_tr.to(device), y_tr.to(device), d_tr_enc.to(device))
    loader = DataLoader(ds_, batch_size=256, shuffle=True)
    hist = []; d_np = d_tr_enc.numpy(); dids = np.unique(d_np)
    for ep in range(epochs):
        enc.train(); vpn_h.train(); el=0; nb=0
        for xb,yb,db in loader:
            feats = enc(xb); loss_c = bce(vpn_h(feats),yb)
            db_np=db.cpu().numpy(); lc=torch.tensor(0.,device=device); np_=0
            for i in range(len(dids)):
                for j in range(i+1,len(dids)):
                    mi=torch.BoolTensor(db_np==dids[i]).to(device)
                    mj=torch.BoolTensor(db_np==dids[j]).to(device)
                    if mi.sum()>5 and mj.sum()>5:
                        lc=lc+coral_loss(feats[mi][:150],feats[mj][:150]); np_+=1
            if np_>0: lc/=np_
            loss=loss_c+lam*lc
            opt.zero_grad(); loss.backward(); opt.step(); el+=loss.item(); nb+=1
        hist.append({"epoch":ep,"loss":el/max(nb,1)})
    return evaluate_model(enc,vpn_h,X_te,y_te,d_te_enc)[0], hist

print("Training CORAL (pooled)...")
coral_res, coral_hist = run_coral(X_tr_t,y_tr_t,d_tr_t,X_te_t,y_te_t,d_te_t)
print(f"  VPN AUC: {coral_res['vpn_auc']:.4f}, Latent Domain AUC: {coral_res['latent_domain_auc']:.4f}")

coral_lodo = []
for test_ds in DATASETS:
    train_ds=[d for d in DATASETS if d!=test_ds]
    tr_mask=(df_scaled["dataset"].isin(train_ds))&(df_scaled["split"]=="train")
    te_mask=df_scaled["dataset"]==test_ds
    if tr_mask.sum()==0 or te_mask.sum()==0: continue
    if len(np.unique(df_scaled.loc[tr_mask,"label"]))<2: continue
    res,_=run_coral(torch.FloatTensor(df_scaled.loc[tr_mask,FEAT_COLS].values),
                     torch.FloatTensor(df_scaled.loc[tr_mask,"label"].values),
                     torch.LongTensor(df_scaled.loc[tr_mask,"ds_enc"].values),
                     torch.FloatTensor(df_scaled.loc[te_mask,FEAT_COLS].values),
                     torch.FloatTensor(df_scaled.loc[te_mask,"label"].values),
                     torch.LongTensor(df_scaled.loc[te_mask,"ds_enc"].values), epochs=100)
    coral_lodo.append({"held_out":test_ds,**res})
    print(f"  CORAL LODO test={test_ds}: vpn_auc={res['vpn_auc']:.4f}")
coral_lodo_df = pd.DataFrame(coral_lodo)

Training CORAL (pooled)...
  VPN AUC: 0.9101, Latent Domain AUC: 0.9991
  CORAL LODO test=iscx: vpn_auc=0.4440
  CORAL LODO test=usbvpn: vpn_auc=0.1850
  CORAL LODO test=vnat: vpn_auc=0.6129


---
## 6. IRM-Inspired Invariant Risk Baseline

In [9]:
def run_irm(X_tr, y_tr, d_tr, X_te, y_te, d_te, lam=1.0, epochs=150):
    # Re-encode domain labels for evaluation
    all_d = torch.cat([d_tr, d_te])
    uniq_d = torch.unique(all_d)
    d_map = {int(v): i for i, v in enumerate(uniq_d)}
    d_te_enc = torch.LongTensor([d_map[int(v)] for v in d_te])

    enc = SharedEncoder(n_features, 32).to(device)
    vpn_h = VPNHead(16).to(device)
    opt = optim.Adam(list(enc.parameters())+list(vpn_h.parameters()), lr=1e-3)
    bce = nn.BCELoss()
    d_np = d_tr.numpy(); dids = np.unique(d_np); hist = []
    X_d = X_tr.to(device); y_d = y_tr.to(device)
    
    for ep in range(epochs):
        enc.train(); vpn_h.train()
        dom_losses = []
        for di in dids:
            m = torch.BoolTensor(d_np==di).to(device)
            if m.sum()<2: continue
            pred = vpn_h(enc(X_d[m]))
            dom_losses.append(bce(pred, y_d[m]))
        if not dom_losses: continue
        mean_l = sum(dom_losses)/len(dom_losses)
        irm_pen = sum((l-mean_l)**2 for l in dom_losses)/len(dom_losses)
        loss = mean_l + lam * irm_pen
        opt.zero_grad(); loss.backward(); opt.step()
        hist.append({"epoch":ep,"loss":loss.item()})
    return evaluate_model(enc,vpn_h,X_te,y_te,d_te_enc)[0], hist

print("Training IRM (pooled)...")
irm_res, irm_hist = run_irm(X_tr_t,y_tr_t,d_tr_t,X_te_t,y_te_t,d_te_t)
print(f"  VPN AUC: {irm_res['vpn_auc']:.4f}, Latent Domain AUC: {irm_res['latent_domain_auc']:.4f}")

irm_lodo = []
for test_ds in DATASETS:
    train_ds=[d for d in DATASETS if d!=test_ds]
    tr_mask=(df_scaled["dataset"].isin(train_ds))&(df_scaled["split"]=="train")
    te_mask=df_scaled["dataset"]==test_ds
    if tr_mask.sum()==0 or te_mask.sum()==0: continue
    if len(np.unique(df_scaled.loc[tr_mask,"label"]))<2: continue
    res,_=run_irm(torch.FloatTensor(df_scaled.loc[tr_mask,FEAT_COLS].values),
                   torch.FloatTensor(df_scaled.loc[tr_mask,"label"].values),
                   torch.LongTensor(df_scaled.loc[tr_mask,"ds_enc"].values),
                   torch.FloatTensor(df_scaled.loc[te_mask,FEAT_COLS].values),
                   torch.FloatTensor(df_scaled.loc[te_mask,"label"].values),
                   torch.LongTensor(df_scaled.loc[te_mask,"ds_enc"].values), epochs=100)
    irm_lodo.append({"held_out":test_ds,**res})
    print(f"  IRM LODO test={test_ds}: vpn_auc={res['vpn_auc']:.4f}")
irm_lodo_df = pd.DataFrame(irm_lodo)

Training IRM (pooled)...
  VPN AUC: 0.5021, Latent Domain AUC: 0.9986
  IRM LODO test=iscx: vpn_auc=0.4229
  IRM LODO test=usbvpn: vpn_auc=0.3353
  IRM LODO test=vnat: vpn_auc=0.2109


---
## 7. Results Consolidation and Training Curves

In [10]:
# Compile all results
all_results = []
for name, pooled, lodo_df in [
    ("DANN", dann_res, dann_lodo_df),
    ("MMD", mmd_res, mmd_lodo_df),
    ("CORAL", coral_res, coral_lodo_df),
    ("IRM", irm_res, irm_lodo_df),
]:
    row = {"method": name, "pooled_vpn_auc": pooled["vpn_auc"],
           "pooled_latent_domain_auc": pooled["latent_domain_auc"]}
    if len(lodo_df) > 0:
        row["lodo_min_auc"] = lodo_df["vpn_auc"].min()
        row["lodo_mean_auc"] = lodo_df["vpn_auc"].mean()
    else:
        row["lodo_min_auc"] = np.nan; row["lodo_mean_auc"] = np.nan
    all_results.append(row)

results_df = pd.DataFrame(all_results)
print("=== Modern Domain Adaptation Results ===")
print(results_df.to_string(index=False))

# Determine verdicts
for i, row in results_df.iterrows():
    lodo_min = row.get("lodo_min_auc", np.nan)
    if np.isnan(lodo_min):
        results_df.loc[i, "verdict"] = "EVALUATION_ERROR"
    elif lodo_min >= 0.65:
        results_df.loc[i, "verdict"] = "MEANINGFUL_TRANSFER_GAIN"
    elif lodo_min >= 0.55:
        results_df.loc[i, "verdict"] = "SLIGHT_IMPROVEMENT_STILL_UNUSABLE"
    else:
        results_df.loc[i, "verdict"] = "NO_IMPROVEMENT"

print("\n=== Verdicts ===")
for _, row in results_df.iterrows():
    print(f"  {row['method']}: {row['verdict']}")

save_csv(results_df, "modern_domain_adaptation_results.csv")

# Latent domain AUC comparison
latent_comp = pd.DataFrame([
    {"representation": "raw_features", "domain_auc": np.nan},  # will be filled
] + [{"representation": r["method"], "domain_auc": r["pooled_latent_domain_auc"]} for _, r in results_df.iterrows()])
save_csv(latent_comp, "latent_domain_auc_comparison.csv")

# Training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (name, hist) in zip(axes.flatten(), [("DANN", dann_hist), ("MMD", mmd_hist), ("CORAL", coral_hist), ("IRM", irm_hist)]):
    hist_df = pd.DataFrame(hist)
    ax.plot(hist_df["epoch"], hist_df["loss"])
    ax.set_title(f"{name} Training Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
plt.tight_layout(); save_fig(fig, "training_curves.png")

=== Modern Domain Adaptation Results ===
method  pooled_vpn_auc  pooled_latent_domain_auc  lodo_min_auc  lodo_mean_auc
  DANN        0.939186                  0.998823      0.215826       0.328789
   MMD        0.924463                  0.999260      0.207489       0.287751
 CORAL        0.910065                  0.999127      0.184968       0.413941
   IRM        0.502092                  0.998565      0.210899       0.323039

=== Verdicts ===
  DANN: NO_IMPROVEMENT
  MMD: NO_IMPROVEMENT
  CORAL: NO_IMPROVEMENT
  IRM: NO_IMPROVEMENT
  ✓ C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\thesis_finalization\nb51_modern_domain_adaptation\modern_domain_adaptation_results.csv
  ✓ C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\thesis_finalization\nb51_modern_domain_adaptation\latent_domain_auc_comparison.csv
  ✓ C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\thesis_finalization\nb51_modern_domain_adaptation\training_curves.png


---
## 8. Final Verdict

In [11]:
best_lodo = results_df["lodo_min_auc"].max()
best_method = results_df.loc[results_df["lodo_min_auc"].idxmax(), "method"] if not results_df["lodo_min_auc"].isna().all() else "NONE"

verdict = {
    "timestamp": TIMESTAMP,
    "methods_tested": list(results_df["method"]),
    "best_method": best_method,
    "best_lodo_min_auc": float(best_lodo) if not np.isnan(best_lodo) else None,
    "any_meaningful_improvement": bool(best_lodo >= 0.65) if not np.isnan(best_lodo) else False,
    "overall_verdict": (
        "MEANINGFUL_TRANSFER_GAIN" if best_lodo >= 0.65 else
        "SLIGHT_IMPROVEMENT_STILL_UNUSABLE" if best_lodo >= 0.55 else
        "NO_MEANINGFUL_IMPROVEMENT"
    ),
    "thesis_implication": (
        f"Modern domain-adaptation methods (DANN, MMD, CORAL, IRM) were tested on the "
        f"21-feature header-only representation. Best LODO min AUC = {best_lodo:.4f} "
        f"({best_method}). {'This represents a meaningful improvement.' if best_lodo >= 0.65 else 'None achieved the 0.65 usability threshold.'} "
        f"This negative result is itself a major finding: it demonstrates that the "
        f"cross-dataset transfer failure is structural, not merely a matter of representation learning."
    ),
    "per_method_verdicts": results_df.set_index("method")["verdict"].to_dict(),
}

save_json(verdict, "notebook51_final_verdict.json")
save_md(f"""# Notebook 51 — Modern Domain Adaptation Summary

## Methods Tested
DANN, MMD, CORAL, IRM

## Best Result
{best_method}: LODO min AUC = {best_lodo:.4f}

## Overall Verdict
**{verdict['overall_verdict']}**

## Per-Method Results
{results_df.to_markdown() if hasattr(results_df,'to_markdown') else results_df.to_string()}

## Thesis Implication
{verdict['thesis_implication']}
""", "notebook51_final_summary.md")

print("\n" + "="*70 + "\nNOTEBOOK 51 COMPLETE\n" + "="*70)

  ✓ C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\thesis_finalization\nb51_modern_domain_adaptation\notebook51_final_verdict.json
  ✓ C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\thesis_finalization\nb51_modern_domain_adaptation\notebook51_final_summary.md

NOTEBOOK 51 COMPLETE
